# Bitcoin 25-Minute Predictor — Offline Training on Snowflake

Runs entirely **without internet**. Everything it needs — dataset, code,
configs — ships inside this repository.

Target container: **4 × NVIDIA A10 (23 GB), 48 vCPU, 100 GB RAM**.

Pipeline: verify compute → verify dataset → distributed training →
walk-forward evaluation → export the winning model.


## 0. Environment

The only thing this needs from the network is Python packages. If the
container has no PyPI access either, vendor wheels into `wheels/` from a
machine that does (`pip download -r requirements.txt -d wheels`) and the
cell below installs from there.


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = Path.cwd()
if (REPO / "notebooks").exists() is False and (REPO.parent / "src").exists():
    REPO = REPO.parent          # running from inside notebooks/
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
print("repo:", REPO)

wheels = REPO / "wheels"
if wheels.exists() and any(wheels.iterdir()):
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                    "--find-links", str(wheels), "-r", "requirements.txt"],
                   check=False)
else:
    print("no local wheels/ -- assuming torch, numpy, pandas are preinstalled")

import torch, numpy, pandas
print("torch", torch.__version__, "| numpy", numpy.__version__)

## 1. Detect available compute

This drives the model sizing, per the project plan: inventory first, then scale up to fit.

In [ ]:
import json
from btcpred.utils.hardware import describe_hardware, autoscale

hw = describe_hardware()
print(json.dumps(hw, indent=2))

rec = autoscale(hw, n_variants=3)
print("\nrecommended sizing:")
print(json.dumps(rec, indent=2))

assert hw["cuda_available"], "No CUDA devices visible -- check the Snowflake compute pool."
N_GPU = len(hw["gpus"])
print(f"\n{N_GPU} GPU(s) ready")

## 2. Dataset

The repo carries the resampled 1-second bar store under `data/bars_1s`
(via Git LFS). If only the raw Binance archives are present, the cell below
rebuilds the bar store from them — still fully offline.


In [ ]:
from btcpred.data.features import BarStore

bars = REPO / "data" / "bars_1s"
raw  = REPO / "data" / "raw"

if not (bars / "meta.json").exists():
    assert raw.exists() and any(raw.rglob("*.zip")), (
        "Neither data/bars_1s nor data/raw is present. "
        "Run scripts/fetch_data.sh on a networked machine and commit the result."
    )
    subprocess.run([sys.executable, "-m", "btcpred.data.resample",
                    "--raw", str(raw), "--out", str(bars)],
                   check=True, env={**os.environ, "PYTHONPATH": str(REPO / "src")})

store = BarStore(bars)
days = len(store) / 86400
print(f"{len(store):,} seconds  ({days:.1f} days)")
print("span:", store.meta["start_ts"], "->", store.meta["start_ts"] + len(store))
print("empty seconds:", f'{store.meta.get("empty_seconds", 0):,}')
assert days > 40, "Need well over 12 h of warm-up plus val/test windows."

## 3. Apply the detected sizing to the config

`scale` widens both the Market Analyser and every Price Predictor to fill the
available VRAM, rather than hard-coding a size that may waste or overflow it.


In [ ]:
cfg_path = REPO / "configs" / "a10x4.json"
cfg = json.loads(cfg_path.read_text())

cfg["data"]["bars"] = str(bars)
cfg["model"]["scale"] = rec["scale"]
cfg["train"]["batch_size"] = rec["batch_size"]
cfg["train"]["amp"] = rec["amp"]
cfg["train"]["grad_checkpoint"] = rec.get("grad_checkpoint", False)
cfg["train"]["workers"] = max(2, (hw["cpu_count"] or 8) // max(N_GPU, 1) - 1)

# Budget: wall-clock is the real constraint, not step count.
cfg["train"]["max_hours"] = 8.0
cfg["train"]["max_steps"] = 200_000

run_cfg = REPO / "configs" / "_active.json"
run_cfg.write_text(json.dumps(cfg, indent=2))
print(json.dumps(cfg["model"], indent=2))
print(json.dumps(cfg["train"], indent=2))

## 4. Distributed training

One shared Market Analyser, three Price Predictor architectures trained
concurrently against it, DDP across all GPUs. There are no epochs: the loop
walks market time forward and is scored every second against recorded reality.

Logs stream below. Expect `pinball` to fall quickly and then flatten — that is
normal, and is *not* yet evidence the model works. Section 5 is the real test.


In [ ]:
env = {**os.environ,
       "PYTHONPATH": str(REPO / "src"),
       "OMP_NUM_THREADS": str(cfg["train"]["workers"]),
       "TOKENIZERS_PARALLELISM": "false"}

cmd = [sys.executable, "-m", "torch.distributed.run",
       f"--nproc_per_node={N_GPU}", "--standalone",
       "-m", "btcpred.train.train_ddp", "--config", str(run_cfg)]
print(" ".join(cmd), flush=True)

proc = subprocess.Popen(cmd, env=env, cwd=REPO, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("exit code:", proc.returncode)

## 5. Walk-forward evaluation and model selection

The decisive column is **skill** (1 − MSE_model / MSE_random-walk). A
25-minute BTC forecast is close to a martingale, so a model can post a tiny
pinball loss simply by predicting "no change". Skill ≤ 0 means exactly that
and the selector will say so rather than hand you a model that looks trained.

**revision_bp** is the stability metric: how far the forecast for a fixed
future instant moves from one second to the next.


In [ ]:
subprocess.run([sys.executable, "-m", "btcpred.train.select_best",
                "--config", str(run_cfg),
                "--ckpt", str(REPO / "checkpoints"),
                "--split", "val", "--step", "30", "--max-points", "4000"],
               check=True, env=env, cwd=REPO)

### Final unbiased check on the test window

Run **once**, after selection is locked. Re-running selection against the test split would leak it.

In [ ]:
subprocess.run([sys.executable, "-m", "btcpred.train.select_best",
                "--config", str(run_cfg),
                "--ckpt", str(REPO / "checkpoints"),
                "--split", "test", "--step", "30", "--max-points", "4000"],
               check=True, env=env, cwd=REPO)

## 6. Export for live use

`analyser.pt`, `best_predictor.pt` and `train_state.json` are everything
`notebooks/molab_live.py` needs. Copy this bundle to molab.


In [ ]:
import shutil, tarfile

ck = REPO / "checkpoints"
bundle = REPO / "artifacts" / "btcpred_live_bundle.tar.gz"
bundle.parent.mkdir(exist_ok=True)

need = ["analyser.pt", "best_predictor.pt", "train_state.json", "metrics.json"]
missing = [f for f in need if not (ck / f).exists()]
assert not missing, f"missing artifacts: {missing}"

with tarfile.open(bundle, "w:gz") as tf:
    for f in need:
        tf.add(ck / f, arcname=f"checkpoints/{f}")

print("wrote", bundle, f"({bundle.stat().st_size/1e6:.1f} MB)")
print(json.dumps(json.loads((ck / "metrics.json").read_text())["metrics"], indent=2))

---

### Before trading any of this

Check, in order:

1. **skill > 0 on the *test* split**, not just validation. If it is not,
   the model has no edge and nothing downstream matters.
2. **revision_bp** small and stable — a jittery signal churns position and
   pays the spread repeatedly.
3. **coverage_80 ≈ 0.80.** Materially below means the uncertainty band is
   overconfident and position sizing built on it will be too large.
4. **net_reward_bp** computed with *your* real fee and slippage, not the
   1 bp placeholder. At a 25-minute horizon on ~17 bp of move, costs
   dominate the result.
